# Les membres statiques

Les membres statiques sont des éléments d'une classe **partagés par toutes les instances**, et non propres à chaque objet.

### Caractéristiques
- Ils ne dépendent **pas** de l'état d'une instance (`self`)
- Ils sont accessibles **sans instancier** la classe : `NomClasse.methode()`
- Utiles pour les **fonctions utilitaires**, **constantes** ou **compteurs partagés**

> ⚠️ Contrairement à C# ou Java, Python permet aussi d'accéder aux membres statiques **depuis une instance** de la classe.

---

## Méthodes statiques avec `@staticmethod`

Le décorateur `@staticmethod` définit une méthode qui **ne reçoit pas** `self` en paramètre. Elle peut être appelée directement sur la classe :

In [ ]:
import datetime

class Voiture:
    @staticmethod
    def verifier_immatriculation(immatriculation):
        return immatriculation.isalnum() and len(immatriculation) == 7

    @staticmethod
    def verifier_annee_fabrication(annee):
        # L'année courante est calculée dynamiquement : pas de valeur codée en dur
        annee_courante = datetime.date.today().year
        return 1900 <= annee <= annee_courante


immatriculation_valide = Voiture.verifier_immatriculation("AB123CD")
annee_valide = Voiture.verifier_annee_fabrication(2010)

print("Immatriculation valide :", immatriculation_valide)
print("Année de fabrication valide :", annee_valide)

---

## Attributs statiques (de classe)

Un attribut défini **au niveau de la classe** (hors constructeur) est partagé par toutes les instances. On y accède via `NomClasse.attribut` :

In [ ]:
class Voiture:
    nombre_total_voitures = 0

    def __init__(self, marque, modele):
        self.marque = marque
        self.modele = modele
        Voiture.nombre_total_voitures += 1

    @staticmethod
    def afficher_nombre_total_voitures():
        print("Nombre total de voitures :", Voiture.nombre_total_voitures)


voiture1 = Voiture("Toyota", "Camry")
voiture2 = Voiture("Honda", "Civic")
voiture3 = Voiture("Ford", "Fusion")

Voiture.afficher_nombre_total_voitures()


---

## `@classmethod` — méthodes de classe

`@classmethod` définit une méthode qui reçoit la **classe** (et non l'instance) comme premier paramètre, conventionnellement appelé `cls`.

| | `@staticmethod` | `@classmethod` |
|---|---|---|
| 1er paramètre | aucun | `cls` (la classe) |
| Accès à la classe | ❌ | ✅ |
| Accès à l'instance | ❌ | ❌ |
| Héritage | non polymorphique | polymorphique (`cls` = sous-classe) |
| Utilisation typique | fonctions utilitaires pures | constructeurs alternatifs, factory methods |

In [ ]:
class Voiture:
    nombre_total = 0

    def __init__(self, marque, modele, annee):
        self.marque = marque
        self.modele = modele
        self.annee = annee
        Voiture.nombre_total += 1   # compteur unique, sur la classe mère

    # Constructeur alternatif via @classmethod
    @classmethod
    def depuis_chaine(cls, chaine):
        """Crée une Voiture depuis une chaîne 'marque,modele,annee'."""
        marque, modele, annee = chaine.split(',')
        return cls(marque.strip(), modele.strip(), int(annee.strip()))
        # cls() → instancie la bonne sous-classe si héritage

    @classmethod
    def afficher_total(cls):
        print(f"Total de {cls.__name__} créées : {cls.nombre_total}")

    @staticmethod
    def valider_annee(annee):
        return 1886 <= annee <= 2100


class VoitureElectrique(Voiture):
    pass
    # ⚠️ Piège : redéclarer ici `nombre_total = 0` créerait un compteur propre
    # à la sous-classe, qui masquerait celui de Voiture — et resterait à 0,
    # car __init__ incrémente toujours Voiture.nombre_total.


v1 = Voiture("Toyota", "Corolla", 2020)
v2 = Voiture.depuis_chaine("Honda, Civic, 2019")
ve1 = VoitureElectrique.depuis_chaine("Tesla, Model 3, 2023")  # cls = VoitureElectrique

Voiture.afficher_total()           # Total de Voiture créées : 3
VoitureElectrique.afficher_total() # Total de VoitureElectrique créées : 3
                                   # (même compteur, hérité de Voiture)

print(Voiture.valider_annee(2024)) # True

---

## `__slots__` — optimisation mémoire

Par défaut, chaque instance Python stocke ses attributs dans un dictionnaire `__dict__`. `__slots__` désactive ce dictionnaire et utilise un tableau fixe à la place, ce qui :
- **Réduit la consommation mémoire** (important pour des milliers d'instances)
- **Accélère** légèrement l'accès aux attributs
- **Interdit** la création d'attributs non déclarés

In [1]:
class PointSansSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class PointAvecSlots:
    __slots__ = ('x', 'y')   # seuls x et y sont autorisés

    def __init__(self, x, y):
        self.x = x
        self.y = y


p1 = PointSansSlots(1, 2)
p2 = PointAvecSlots(1, 2)

p1.z = 3           # OK — __dict__ permet d'ajouter des attributs
# p2.z = 3         # AttributeError — z n'est pas dans __slots__

print(hasattr(p1, '__dict__'))  # True
print(hasattr(p2, '__dict__'))  # False

# Comparaison mémoire : la taille exacte varie selon la version de Python,
# on la mesure donc plutôt que de citer un chiffre figé
import sys
print(f"Taille du __dict__ de p1 : {sys.getsizeof(p1.__dict__)} octets")
# p2 n'a pas de __dict__ → beaucoup plus compact

True
False
Taille du __dict__ de p1 : 296 octets
